# Source review — national-CERT advisory feeds (ACSC / NCSC / CCCS)

The benchmark's report-driven tasks (ATE, TAA, SYN) currently draw from **CISA's AA-series**
(`fetch_aa_advisory_index` + `parse_advisory_page`). This notebook reviews three national-CERT
feeds as **additional advisory sources** and scores each against what the ingest path actually
needs — *not* against how authoritative they are (all three are top-tier), but against how well
they fit the pipeline.

Candidates:
- **ACSC** (Australia) — `cyber.gov.au/about-us/view-all-content/alerts-and-advisories`
- **NCSC** (UK) — `ncsc.gov.uk/section/keep-up-to-date/reports-advisories`
- **CCCS** (Canada) — `cyber.gc.ca/en/alerts-advisories`

The findings cells below are **captured from a live structural probe** (June 2026); the optional
probe in §3 re-fetches one sample page per source so you can re-verify. The notebook is DB-free.

In [ ]:
import os, sys, json

def _find_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "src", "glokta")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (a dir containing src/glokta)")

ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

# Best-effort load of the repo .env so live HF calls have HF_TOKEN; no dotenv dependency.
_envp = os.path.join(ROOT, ".env")
if os.path.exists(_envp):
    for _line in open(_envp):
        _s = _line.strip()
        if _s and not _s.startswith("#") and "=" in _s:
            _k, _v = _s.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("TESTING", "1")  # relax settings validators if .env is absent

MODEL = "huggingface/meta-llama/Llama-3.1-8B-Instruct"
LIVE = bool(os.environ.get("HF_TOKEN"))
print("repo root :", ROOT)
print("model     :", MODEL)
print("LIVE calls:", LIVE, "(set HF_TOKEN to enable real inference)")

def run_model(prompt, canned, max_tokens=256):
    """Call the model live if HF_TOKEN is set, else return a canned example response."""
    if LIVE:
        from glokta.infrastructure.cti.inference import complete
        try:
            return complete(MODEL, prompt, max_tokens=max_tokens, timeout=60.0, max_retries=1)
        except Exception as exc:
            print("[live call failed -> canned]", type(exc).__name__, str(exc)[:80])
            return canned
    print("[offline -> canned response]")
    return canned

## 1. What the benchmark needs from a source

Suitability is defined by the existing ingest path, not by editorial quality. A source must give us:

| # | Criterion | Why (which code depends on it) |
|---|-----------|--------------------------------|
| C1 | **Pageable index** | `fetch_aa_advisory_index` walks a listing to enumerate advisory URLs newest-first. |
| C2 | **On-page section structure** | `parse_advisory_page` isolates a body container and splits on h2/h3 headings; PDF-only advisories don't parse. |
| C3 | **Separable conclusion labels** | ATE/TAA/SYN labels = ATT&CK technique-ids, actor attribution, CVEs, IOCs — and `reconstruct_inputs`/`mask_conclusions` need them in their *own* sections so they can be withheld. |
| C4 | **Original (non-joint) content** | Co-sealed joint advisories duplicate the CISA AA corpus (same text, different slug) → contamination + dedup. |
| C5 | **Volume & recency** | Enough recent original items to be worth a connector. |
| C6 | **Fetchability** | Static HTML (cheap `httpx`) vs JS/bot-throttled (needs headless). |

## 2. Captured findings
One sample advisory per source was parsed live to capture its real heading set and label availability. `headings_verified=False` means the page/listing was bot-throttled during this review and the structure is indicative only.

In [ ]:
SOURCES = {
    "ACSC": {
        "index_url": "https://www.cyber.gov.au/about-us/view-all-content/alerts-and-advisories?type=329",
        "index_pagination": "JS-rendered facet listing; bot-throttled (httpx timed out repeatedly)",
        "advisory_url_patterns": ["/about-us/view-all-content/alerts-and-advisories/<slug>",
                                  "/about-us/advisories/<slug>"],
        "sample": "apt40-advisory-prc-mss-tradecraft-in-action",
        # Indicative only — the page itself bot-throttled in this review.
        "headings": ["Summary", "Technical details", "Detection and mitigation",
                     "Indicators of compromise", "MITRE ATT&CK"],
        "headings_verified": False,
        "has_attack_ids": True, "has_actor_attribution": True,
        "has_cves": True, "has_iocs": True,
        "content_on_page": "mixed — several advisories are PDF-only (e.g. ACSC-Advisory-2020-008)",
        "joint_overlap": "HIGH — flagship advisories (APT40, Salt Typhoon) are co-sealed with CISA/FBI/NCSC",
        "fetchability": "POOR — listing + pages bot-throttled; needs headless or a facet/JSON API",
    },
    "NCSC": {
        "index_url": "https://www.ncsc.gov.uk/section/keep-up-to-date/reports-advisories",
        "index_pagination": "shallow single list (~6 items), no pager; mixes news posts and advisories",
        "advisory_url_patterns": ["/news/<slug>"],
        "sample": "apt28-exploit-routers-to-enable-dns-hijacking-operations",
        "headings": ["Executive summary", "Introduction", "APT28 malicious DNS activity",
                     "Indicators of compromise", "MITRE ATT&CK®", "Mitigation"],
        "headings_verified": True,
        "has_attack_ids": True, "has_actor_attribution": True,
        "has_cves": True, "has_iocs": True,
        "content_on_page": "yes on flagship advisories; many full CSAs + malware reports are PDF",
        "joint_overlap": "HIGH — most CSAs co-sealed with CISA/allies",
        "fetchability": "OK for /news/<slug> pages; index too shallow to enumerate a corpus",
    },
    "CCCS": {
        "index_url": "https://www.cyber.gc.ca/en/alerts-advisories",
        "index_pagination": "single very large listing page (one fetch enumerates all); bilingual /en/ /fr/",
        "advisory_url_patterns": ["/en/alerts-advisories/<slug>", "/en/alerts/<slug>"],
        "sample": "alphvblackcat-ransomware-targeting-canadian-industries",
        "headings": ["Audience", "Purpose", "Details",
                     "Tactics, techniques, and procedures (TTP)", "Suggested actions",
                     "Indicators of compromise", "References", "MITRE ATT&CK techniques"],
        "headings_verified": True,
        "has_attack_ids": True, "has_actor_attribution": True,
        "has_cves": False,  # ALPHV sample had none; CCCS CVE coverage is inconsistent
        "has_iocs": True,
        "content_on_page": "yes — full text on-page (AL-numbered originals, e.g. AL23-010), no PDF",
        "joint_overlap": "MIXED — original AL-numbered items + republished CERT-FR/CISA items",
        "fetchability": "GOOD per-page; index page is huge (>10MB) but a single fetch",
    },
}
for name, s in SOURCES.items():
    print(f"{name}: verified_headings={s['headings_verified']}  "
          f"attack={s['has_attack_ids']} actor={s['has_actor_attribution']} "
          f"cve={s['has_cves']} ioc={s['has_iocs']}")

### Does our existing CISA section-classifier transfer?

`parse_advisory_page` buckets each heading via `_classify_heading` into **Overview / Technical
Details / Indicators of Compromise** (kept as model input) vs **Dropped** (conclusions + chrome).
Running it over each source's *real* heading set shows what the current rules already handle and
where a source needs tuning before its conclusions are safely withheld.

In [ ]:
from glokta.infrastructure.cti.connectors.report import _classify_heading
from glokta.infrastructure.cti.claim_extraction import _INPUT_SECTIONS

print(f"{'source':6} {'heading':42} {'kept?':5} bucket")
print("-" * 78)
for name, s in SOURCES.items():
    for h in s["headings"]:
        keep, bucket = _classify_heading(h)
        flag = "KEEP " if keep else "drop "
        print(f"{name:6} {h[:42]:42} {flag} {bucket}")
    print()

**Reading it.** A heading that states a *conclusion* must land in **Dropped**; an *observation*
heading should map to one of `_INPUT_SECTIONS`. Watch for two failure modes:

- **Leak risk** — a conclusion/chrome heading that gets KEPT (any heading the rules don't recognise
  defaults to *Technical Details*). Empirically, CCCS *"Tactics, techniques, and procedures (TTP)"*,
  *"Suggested actions"* and *"Audience"* are all **kept** today — the first is a conclusion section
  that would leak its TTP narrative into the inputs, the other two are chrome. Only CCCS *"MITRE
  ATT&CK techniques"*, *"Purpose"* and *"References"* drop correctly. Fixing CCCS means adding
  `tactics`/`ttp`/`suggested action`/`audience` to `_DROP_KEYWORDS`. (The NCSC set, and the
  indicative ACSC set, classify cleanly — every conclusion/chrome heading already drops.)
- **Lost evidence** — an observation heading wrongly dropped (none observed here).

So the generic parser transfers structurally, but **each source needs a small drop-keyword /
body-container patch** before its conclusions are reliably withheld. That patch is the real cost
of adding a source.

## 3. Optional live structural probe
Re-fetch one sample advisory per source and run the generic body parser + label regexes over it. Off by default (ACSC throttles); set `SOURCE_REVIEW_PROBE=1` to enable.

In [ ]:
PROBE = os.environ.get("SOURCE_REVIEW_PROBE", "0") == "1"
HEADERS = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36"}

if PROBE:
    import httpx
    from glokta.infrastructure.cti.connectors.report import (
        _AdvisorySectionParser, _advisory_body, _TECHNIQUE_RE, _CVE_RE)
    base = {"ACSC": "https://www.cyber.gov.au/about-us/view-all-content/alerts-and-advisories/",
            "NCSC": "https://www.ncsc.gov.uk/news/",
            "CCCS": "https://www.cyber.gc.ca/en/alerts-advisories/"}
    with httpx.Client(timeout=30.0, headers=HEADERS, follow_redirects=True) as c:
        for name, s in SOURCES.items():
            url = base[name] + s["sample"]
            try:
                html = c.get(url).text
                parser = _AdvisorySectionParser(); parser.feed(_advisory_body(html))
                heads = [h for h, _ in parser.result() if h]
                techs = sorted({t.upper() for t in _TECHNIQUE_RE.findall(html)})
                cves = sorted({c.upper() for c in _CVE_RE.findall(html)})
                print(f"{name}: {len(heads)} headings, {len(techs)} ATT&CK ids, {len(cves)} CVEs")
                print("   headings:", heads[:8])
            except Exception as exc:
                print(f"{name}: fetch failed -> {type(exc).__name__} {str(exc)[:60]}")
else:
    print("probe disabled (set SOURCE_REVIEW_PROBE=1). Using captured findings from §2.")

## 4. Suitability scorecard
Score each criterion 0 (blocker) / 1 (needs work) / 2 (good). The total is a *fit* score for the ingest path, weighted toward index + fetchability + non-overlap (the things that cost connector work), since label/structure quality is uniformly high.

In [ ]:
import pandas as pd

# (C1 index, C2 on-page structure, C3 labels, C4 originality/non-overlap, C5 volume, C6 fetchability)
SCORES = {
    "ACSC": {"C1 index": 0, "C2 structure": 1, "C3 labels": 2, "C4 non-overlap": 0, "C5 volume": 2, "C6 fetch": 0},
    "NCSC": {"C1 index": 1, "C2 structure": 2, "C3 labels": 2, "C4 non-overlap": 0, "C5 volume": 1, "C6 fetch": 1},
    "CCCS": {"C1 index": 2, "C2 structure": 2, "C3 labels": 2, "C4 non-overlap": 1, "C5 volume": 2, "C6 fetch": 2},
}
df = pd.DataFrame(SCORES).T
df["TOTAL"] = df.sum(axis=1)
df = df.sort_values("TOTAL", ascending=False)
print(df.to_string())
print("\nmax possible:", 6 * 2)
print("ranking     :", " > ".join(df.index))

## 5. Findings & recommendation

**All three publish genuinely high-quality, well-structured advisories** — ATT&CK ids, attributed
actors (with aliases), and IOCs are present on the flagship pages of every source. The differences
that matter are all about *ingest fit*, and they split the three cleanly:

- **CCCS (Canada) — adopt first.** Single-fetch index, full text on-page (no PDF), consistent
  `AL`-numbered originals, clean heading structure. The only work: dedup republished CERT-FR/CISA
  items, add `tactics`/`ttp`/`suggested action`/`audience` to `_DROP_KEYWORDS` (verified in §2:
  these three CCCS sections are currently kept and would leak), and handle the `/en/alerts/` vs
  `/en/alerts-advisories/` URL split. CVE coverage is inconsistent, so it helps ATE/
  TAA/SYN more than Forecast.
- **NCSC (UK) — secondary.** Individual `/news/<slug>` pages parse beautifully, but the HTML index
  is too shallow to enumerate a corpus (no pager, ~6 mixed news/advisory items) and many full CSAs
  are PDF-only. Worth a thin connector for the on-HTML flagship advisories once an enumeration path
  (sitemap or search API) is found.
- **ACSC (Australia) — defer.** Highest friction: the facet listing and the advisory pages are
  bot-throttled (every `httpx` fetch timed out in this review), several advisories are PDF-only, and
  the flagship items are co-sealed with CISA. Needs a headless fetcher or an undocumented JSON facet
  endpoint before it's worth the connector.

**The cross-cutting caveat — joint-advisory overlap (C4).** All three co-seal joint advisories with
CISA, so a large share of their output is *the same text already in our AA-series corpus*, under a
different slug. Ingesting those naively would (a) double-count items and (b) defeat contamination
control, since the model may have trained on the CISA version. **Before adding any of these sources,
the ingest path needs a cross-source dedup step** (content-hash or normalised-title match against
existing `cti_items`), keeping only each source's *original* national content. That dedup — plus a
per-source body-container + drop-keyword patch to `parse_advisory_page` — is the actual engineering
cost; the parser/claim-extraction cores transfer unchanged.

**Recommendation:** add **CCCS** next (best fit, lowest cost), with a content-hash dedup gate;
treat **NCSC** as a follow-on pending an enumeration path; **defer ACSC** until a non-throttled
fetch route exists.